# 02 - Build our Darija tokenizer

We will build a simple word tokenizer, look at how it handles unknown words, and then train a **ByteLevel BPE** tokenizer on the data prepared in notebook 01.

The vocabulary and merges are learned from the training split only. We do not load a pretrained tokenizer. The `tokenizers` library supplies the BPE algorithm; we will build the GPT model later.

This notebook is delivered unexecuted. The training cell only runs when you execute it. Short phrases in the notebook are teaching examples, not AtlasIA data.

In [1]:
import hashlib
import json
import re
import unicodedata
from pathlib import Path

import tokenizers
from tokenizers import Tokenizer, decoders, models, normalizers
from tokenizers import pre_tokenizers, trainers

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / 'README.md').is_file() and (p / 'notebooks').is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError('Open this notebook from the darija-llm-lab project directory.')
PROCESSED_DIR = ROOT / 'data' / 'processed'
TOKENIZER_DIR = ROOT / 'artifacts' / 'tokenizer'
VOCAB_SIZE = 2000
MIN_FREQUENCY = 2
SPECIAL_TOKENS = ['[UNK]', '[PAD]', '[BOS]', '[EOS]']
print('Tokenizers version:', tokenizers.__version__)
print('Target vocabulary size:', VOCAB_SIZE)

Tokenizers version: 0.22.1
Target vocabulary size: 2000


## 1. Tokens, IDs, and vocabulary

A **token** is a piece of text; an **ID** is its number in the vocabulary. This number does not express the token's meaning or importance. A future GPT will map these IDs to vectors called embeddings.

For our first experiment, we split word-like character sequences and isolate punctuation. `encode` maps the units to IDs; `decode` performs the reverse operation.

In [2]:
def normalize(text):
    return unicodedata.normalize('NFC', text)

class WordTokenizer:
    def __init__(self, texts):
        pieces = {
            piece for text in texts
            for piece in re.findall(r'\w+|[^\w\s]', normalize(text))
        }
        self.vocab = {'[UNK]': 0}
        self.vocab.update({piece: i + 1 for i, piece in enumerate(sorted(pieces))})
        self.inverse = {i: piece for piece, i in self.vocab.items()}

    def encode(self, text):
        pieces = re.findall(r'\w+|[^\w\s]', normalize(text))
        return [self.vocab.get(piece, 0) for piece in pieces]

    def decode(self, ids):
        return ' '.join(self.inverse[i] for i in ids)

toy = WordTokenizer(['salam, labas?'])
print('Vocabulary:', toy.vocab)
print('IDs:', toy.encode('salam, labas?'))
print('Decoded text:', toy.decode(toy.encode('salam, labas?')))
print('Unknown word:', toy.encode('salam khouya'))

Vocabulary: {'[UNK]': 0, ',': 1, '?': 2, 'labas': 3, 'salam': 4}
IDs: [4, 1, 3, 2]
Decoded text: salam , labas ?
Unknown word: [4, 0]


The word-based version replaces an unknown unit with `[UNK]` and puts spaces back between units during decoding. It can lose the original word and formatting. It is a teaching example, not a way to preserve arbitrary Unicode text.

## 2. The BPE idea

A simplified example: two documents contain `abab`, and one document contains `ab`. The pair `(a, b)` appears five times; `(b, a)` appears twice. The first merge creates `ab`. The documents become `[ab, ab]`, `[ab, ab]`, and `[ab]`. We then count pairs again.

Training records units and a merge order. Encoding new text applies those rules without learning them again. BPE learns frequencies, not semantic relationships.

Our **ByteLevel** variant starts from UTF-8 bytes. An Arabic letter can use several bytes: a byte, a character, and a token are not the same thing. The initial 256-byte alphabet can represent new text without losing ordinary symbols.

## 3. Load the splits from notebook 01

We verify the text hashes and check that there are no exact duplicates between the splits. The tokenizer is trained **only on `train_texts`**. The validation split is reserved for measurements after training.

In [3]:
def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f'{path} is missing: run notebook 01.')
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()
            if line.strip()]
    for row in rows:
        actual_hash = hashlib.sha256(row['text'].encode('utf-8')).hexdigest()
        if actual_hash != row['sha256']:
            raise ValueError('The text has changed since preprocessing.')
    return rows

train_path = PROCESSED_DIR / 'train.jsonl'
valid_path = PROCESSED_DIR / 'validation.jsonl'
train_rows = read_jsonl(train_path)
valid_rows = read_jsonl(valid_path)
train_texts = [row['text'] for row in train_rows]
valid_texts = [row['text'] for row in valid_rows]
if not train_texts or not valid_texts:
    raise ValueError('Both splits must contain documents.')
assert set(train_texts).isdisjoint(valid_texts)
corpus_manifest = json.loads(
    (PROCESSED_DIR / 'corpus_manifest.json').read_text(encoding='utf-8')
)
print('Training:', len(train_texts), '| Validation:', len(valid_texts))

Training: 900 | Validation: 100


## 4. Configure the tokenizer before training

- **NFC**: the same normalization convention as notebook 01.
- **ByteLevel**: byte representation and pre-tokenization; no artificial leading space is added.
- **BpeTrainer**: learns merges from the text. The target vocabulary size includes the alphabet and special tokens.
- **ByteLevel decoder**: reconstructs normalized text from a sequence of IDs.

Special tokens: `[UNK]` for an unknown unit, `[PAD]` for padding a sequence, `[BOS]` for a beginning, and `[EOS]` for an end. Reserving them does not insert them automatically. In our pipeline, EOS will be added explicitly by ID in notebook 03.

In [4]:
def build_bpe(vocab_size=2000, min_frequency=2):
    if vocab_size < 256 + len(SPECIAL_TOKENS):
        raise ValueError('The 256 bytes and special tokens must be covered.')
    tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=min_frequency,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=SPECIAL_TOKENS, show_progress=True,
    )
    return tokenizer, trainer

## 5. Train the merges

**Running this cell starts training.** First check the corpus and settings. It trains on neither the invented examples above nor the validation split.

In Tokenizers 0.22.1, `encode_special_tokens=True` lets literal strings such as `[EOS]` in the text remain ordinary text. This avoids confusing them with markers that we add by ID.

In [5]:
tokenizer, trainer = build_bpe(VOCAB_SIZE, MIN_FREQUENCY)
tokenizer.train_from_iterator(
    train_texts, trainer=trainer, length=len(train_texts)
)
tokenizer.encode_special_tokens = True
print('Actual vocabulary size:', tokenizer.get_vocab_size())
print('Special tokens:', {s: tokenizer.token_to_id(s) for s in SPECIAL_TOKENS})

Actual vocabulary size: 2000
Special tokens: {'[UNK]': 0, '[PAD]': 1, '[BOS]': 2, '[EOS]': 3}


## 6. Inspect tokens and verify decoding

`encoding.tokens` shows ByteLevel's internal representations; some may look unusual. To recover the text, decode the **whole sequence**, not each token separately. A token may contain only part of the bytes for an Arabic letter.

The expected invariant is `decode(encode(text)) == NFC(text)`. It applies to normalized text because normalization can change the Unicode representation.

In [6]:
examples = [
    'سلام، لاباس؟', 'salam, labas? wach kolchi mzyan?',
    'ghadi nji demain.', 'cafe\u0301', 'salam  labas\n🙂',
    'literal text [UNK] [PAD] [BOS] [EOS]', '',
]
unk_id = tokenizer.token_to_id('[UNK]')
for text in examples + valid_texts:
    encoding = tokenizer.encode(text, add_special_tokens=False)
    decoded = tokenizer.decode(encoding.ids, skip_special_tokens=False)
    assert decoded == normalize(text), (text, decoded)
    assert unk_id not in encoding.ids
print('Round trips validated:', len(examples) + len(valid_texts))

text = examples[0]
encoding = tokenizer.encode(text, add_special_tokens=False)
print('Text:', text)
print('Internal tokens:', encoding.tokens)
print('IDs:', encoding.ids)
print('Decoded text:', tokenizer.decode(encoding.ids))

Round trips validated: 107
Text: سلام، لاباس؟
Internal tokens: ['Ø³', 'ÙĦØ§Ùħ', 'Ø', 'Į', 'ĠÙĦØ§Ø¨Ø§Ø³', 'Ø', 'Ł']
IDs: [292, 762, 152, 238, 1859, 152, 257]
Decoded text: سلام، لاباس؟


## 7. Measure fragmentation on validation

Zero `[UNK]` tokens checks coverage, not language understanding. We also measure tokens per 100 characters. Python counts Unicode code points here, which are not always the same as visible characters.

A larger vocabulary can produce shorter sequences, but it also enlarges the model's future embedding table. Compare choices on the same validation documents; do not claim one size is optimal without measuring it.

In [7]:
valid_encodings = [tokenizer.encode(text, add_special_tokens=False)
                   for text in valid_texts]
n_tokens = sum(len(item.ids) for item in valid_encodings)
n_chars = sum(len(normalize(text)) for text in valid_texts)
evaluation = {
    'validation_documents': len(valid_texts),
    'validation_tokens': n_tokens,
    'tokens_per_100_characters': round(100 * n_tokens / max(1, n_chars), 2),
    'unknown_tokens': sum(item.ids.count(unk_id) for item in valid_encodings),
    'roundtrip_documents_checked': len(valid_texts),
}
print(json.dumps(evaluation, indent=2, ensure_ascii=False))

{
  "validation_documents": 100,
  "validation_tokens": 48885,
  "tokens_per_100_characters": 36.67,
  "unknown_tokens": 0,
  "roundtrip_documents_checked": 100
}


## 8. Save and reload

The JSON file stores the tokenizer components, vocabulary, and merges. With this library version, `encode_special_tokens=True` must be **restored after reloading**.

Outputs go to `artifacts/tokenizer/`, which is excluded from Git. The file appearing after you run the notebook represents the tokenizer actually trained; the notebooks are delivered without a trained tokenizer.

In [8]:
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tokenizer_path = TOKENIZER_DIR / 'tokenizer_darija.json'
tokenizer.save(str(tokenizer_path))
loaded = Tokenizer.from_file(str(tokenizer_path))
loaded.encode_special_tokens = True
for text in examples + valid_texts:
    assert loaded.encode(text).ids == tokenizer.encode(text).ids

training_manifest = {
    'dataset': corpus_manifest['dataset'],
    'corpus': corpus_manifest,
    'fit_partition': 'train', 'validation_used_for_training': False,
    'train_documents': len(train_texts), 'validation_documents': len(valid_texts),
    'train_jsonl_sha256': hashlib.sha256(train_path.read_bytes()).hexdigest(),
    'validation_jsonl_sha256': hashlib.sha256(valid_path.read_bytes()).hexdigest(),
    'requested_vocabulary': VOCAB_SIZE,
    'actual_vocabulary': tokenizer.get_vocab_size(),
    'min_frequency': MIN_FREQUENCY, 'normalization': 'NFC',
    'special_token_ids': {s: tokenizer.token_to_id(s) for s in SPECIAL_TOKENS},
    'literal_special_tokens_as_text': True,
    'tokenizers_version': tokenizers.__version__,
    'tokenizer_sha256': hashlib.sha256(tokenizer_path.read_bytes()).hexdigest(),
}
for name, content in [('training_manifest.json', training_manifest),
                      ('evaluation.json', evaluation)]:
    (TOKENIZER_DIR / name).write_text(
        json.dumps(content, ensure_ascii=False, indent=2), encoding='utf-8'
    )
print('Tokenizer and metrics saved to:', TOKENIZER_DIR)

Tokenizer and metrics saved to: C:\Users\sail_\Documents\Codex\2026-10-03\https-github-com-rasbt-llms-from\outputs\darija-llm-lab\artifacts\tokenizer
